# 🚀 RLHF Phi-3 Pipeline: Setup and Configuration

Welcome to the RLHF Phi-3 Pipeline! This notebook will guide you through setting up your environment and configuring the pipeline for training.

## 📋 What You'll Learn

- ✅ Set up Google Colab environment with GPU access
- ✅ Configure Google Drive for checkpoint persistence
- ✅ Set up authentication for Weights & Biases and HuggingFace
- ✅ Install and configure the RLHF pipeline
- ✅ Understand configuration options and validation
- ✅ Run a quick system check and demo

## ⏱️ Estimated Time: 15-20 minutes

## 🔧 Step 1: Environment Setup

First, let's check our environment and set up the necessary components.

In [ ]:
# Check if we're running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("📍 Running in local environment")

# Check GPU availability
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 GPU Available: {gpu_name} ({gpu_memory:.1f}GB)")
else:
    print("⚠️  No GPU detected. This pipeline requires a GPU for training.")
    print("   In Colab: Runtime > Change runtime type > Hardware accelerator > GPU")

## 💾 Step 2: Google Drive Setup (Colab Only)

We'll mount Google Drive to persist checkpoints across sessions.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    import os
    
    # Mount Google Drive
    print("📁 Mounting Google Drive...")
    drive.mount('/content/drive')
    
    # Create project directory
    project_dir = "/content/drive/MyDrive/rlhf-phi3"
    os.makedirs(project_dir, exist_ok=True)
    os.makedirs(f"{project_dir}/checkpoints", exist_ok=True)
    os.makedirs(f"{project_dir}/logs", exist_ok=True)
    
    print(f"✅ Project directory created: {project_dir}")
    
    # Check available space
    import shutil
    total, used, free = shutil.disk_usage("/content/drive")
    print(f"💾 Google Drive space: {free / 1e9:.1f}GB free / {total / 1e9:.1f}GB total")
    
    if free < 10e9:  # Less than 10GB
        print("⚠️  Warning: Low disk space. Consider cleaning up Google Drive.")
else:
    project_dir = "./rlhf-phi3"
    print(f"📁 Using local project directory: {project_dir}")

## 📦 Step 3: Install Dependencies

Install the RLHF pipeline and all required dependencies.

In [ ]:
if IN_COLAB:
    # Clone the repository
    !git clone https://github.com/your-username/rlhf-phi3-pipeline.git
    %cd rlhf-phi3-pipeline
    
    # Install the package and dependencies
    !pip install -q -r requirements.txt
    !pip install -q -e .
    
    print("✅ Installation completed!")
else:
    print("📍 For local installation, run:")
    print("   git clone https://github.com/your-username/rlhf-phi3-pipeline.git")
    print("   cd rlhf-phi3-pipeline")
    print("   pip install -r requirements.txt")
    print("   pip install -e .")

## 🔐 Step 4: Authentication Setup

Set up authentication for Weights & Biases (experiment tracking) and HuggingFace (model publishing).

In [ ]:
# Weights & Biases setup
print("🔐 Setting up Weights & Biases...")
print("1. Go to https://wandb.ai/ and create a free account")
print("2. Go to https://wandb.ai/authorize to get your API key")
print("3. Run the cell below and paste your API key when prompted")

import wandb
try:
    wandb.login()
    print("✅ Weights & Biases authentication successful!")
except Exception as e:
    print(f"❌ W&B authentication failed: {e}")
    print("   You can continue without W&B, but experiment tracking will be disabled.")

In [ ]:
# HuggingFace setup
print("🤗 Setting up HuggingFace...")
print("1. Go to https://huggingface.co/ and create a free account")
print("2. Go to https://huggingface.co/settings/tokens to create an access token")
print("3. Run the cell below and paste your token when prompted")

from huggingface_hub import login
try:
    login()
    print("✅ HuggingFace authentication successful!")
except Exception as e:
    print(f"❌ HuggingFace authentication failed: {e}")
    print("   You can continue without HF authentication, but model publishing will be disabled.")

## ⚙️ Step 5: Configuration Setup

Load and customize the pipeline configuration.

In [ ]:
from rlhf_phi3 import Config
import yaml

# Load the Colab-optimized configuration
config_path = "configs/colab_config.yaml" if IN_COLAB else "configs/default_config.yaml"
config = Config.from_yaml(config_path)

print(f"📋 Loaded configuration from: {config_path}")
print(f"🤖 Model: {config.model.model_name}")
print(f"🎯 Max length: {config.model.max_length}")
print(f"🔧 LoRA rank: {config.lora.r}")
print(f"📊 Batch size: {config.training.batch_size}")

# Update paths for current environment
if IN_COLAB:
    config.paths.base_output_dir = project_dir
    config.paths.checkpoint_dir = f"{project_dir}/checkpoints"
    config.paths.logs_dir = f"{project_dir}/logs"

print(f"💾 Output directory: {config.paths.base_output_dir}")

## ✅ Step 6: Configuration Validation

Validate the configuration to ensure all parameters are correct.

In [ ]:
# Validate configuration
print("🔍 Validating configuration...")
validation_errors = config.validate()

if validation_errors:
    print("❌ Configuration validation failed:")
    for error in validation_errors:
        print(f"   • {error}")
else:
    print("✅ Configuration validation passed!")

# Display key configuration details
print("\n📊 Configuration Summary:")
print(f"   Model: {config.model.model_name}")
print(f"   Context Length: {config.model.max_length}")
print(f"   LoRA Rank: {config.lora.r}")
print(f"   LoRA Alpha: {config.lora.alpha}")
print(f"   Batch Size: {config.training.batch_size}")
print(f"   Learning Rate: {config.training.learning_rate}")
print(f"   SFT Epochs: {config.training.sft_epochs}")
print(f"   PPO Steps: {config.training.ppo_steps}")

## 🧪 Step 7: System Check

Run a comprehensive system check to ensure everything is working correctly.

In [ ]:
import psutil
import torch
from transformers import AutoTokenizer

print("🔍 Running system check...\n")

# Check GPU
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} ({gpu_memory:.1f}GB)")
    
    # Check GPU memory usage
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"   Memory: {allocated:.1f}GB allocated, {reserved:.1f}GB reserved")
else:
    print("❌ GPU: Not available")

# Check system memory
memory = psutil.virtual_memory()
print(f"✅ RAM: {memory.total / 1e9:.1f}GB total, {memory.available / 1e9:.1f}GB available")

# Check disk space
disk = shutil.disk_usage("/content" if IN_COLAB else ".")
print(f"✅ Disk: {disk.free / 1e9:.1f}GB free / {disk.total / 1e9:.1f}GB total")

# Test model loading
print("\n🤖 Testing model components...")
try:
    tokenizer = AutoTokenizer.from_pretrained(config.model.model_name)
    print(f"✅ Tokenizer loaded: {len(tokenizer)} tokens")
except Exception as e:
    print(f"❌ Tokenizer loading failed: {e}")

# Test imports
print("\n📦 Testing package imports...")
try:
    from rlhf_phi3 import TrainingOrchestrator, DatasetManager, ModelManager
    print("✅ Core components imported successfully")
except Exception as e:
    print(f"❌ Import failed: {e}")

print("\n🎉 System check completed!")

## 🚀 Step 8: Quick Demo

Run a quick demo to verify everything is working correctly.

In [ ]:
from rlhf_phi3 import TrainingOrchestrator

print("🚀 Running quick demo...")

# Create a demo configuration with minimal settings
demo_config = config.copy()
demo_config.datasets.max_samples = 100  # Use only 100 samples
demo_config.training.sft_epochs = 1     # Single epoch
demo_config.training.ppo_steps = 10     # Minimal PPO steps

# Initialize the orchestrator
orchestrator = TrainingOrchestrator(demo_config)

print("✅ Training orchestrator initialized successfully!")
print(f"📊 Demo configuration:")
print(f"   Max samples: {demo_config.datasets.max_samples}")
print(f"   SFT epochs: {demo_config.training.sft_epochs}")
print(f"   PPO steps: {demo_config.training.ppo_steps}")

# Test dataset loading
print("\n📚 Testing dataset loading...")
try:
    dataset_manager = orchestrator.dataset_manager
    sft_dataset = dataset_manager.load_sft_dataset(streaming=True)
    print(f"✅ SFT dataset loaded successfully")
    
    # Show a sample
    sample = next(iter(sft_dataset))
    print(f"   Sample keys: {list(sample.keys())}")
except Exception as e:
    print(f"❌ Dataset loading failed: {e}")

print("\n🎉 Demo completed successfully!")
print("\n➡️  You're ready to proceed to the training tutorials!")

## 💾 Step 9: Save Configuration

Save your configuration for use in subsequent notebooks.

In [ ]:
# Save the validated configuration
config_save_path = f"{project_dir}/my_config.yaml"
config.save_yaml(config_save_path)

print(f"💾 Configuration saved to: {config_save_path}")
print("\n📋 You can load this configuration in other notebooks with:")
print(f'   config = Config.from_yaml("{config_save_path}")')

# Create a summary file
summary = f"""
# RLHF Phi-3 Pipeline Setup Summary

## Environment
- Platform: {'Google Colab' if IN_COLAB else 'Local'}
- GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}
- Project Directory: {project_dir}

## Configuration
- Model: {config.model.model_name}
- LoRA Rank: {config.lora.r}
- Batch Size: {config.training.batch_size}
- Learning Rate: {config.training.learning_rate}

## Next Steps
1. Run 02_sft_training_tutorial.ipynb for supervised fine-tuning
2. Run 03_reward_model_tutorial.ipynb for reward model training
3. Run 04_ppo_training_tutorial.ipynb for PPO training
4. Run 05_evaluation_and_publishing.ipynb for evaluation and publishing

Setup completed on: {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

with open(f"{project_dir}/setup_summary.md", "w") as f:
    f.write(summary)

print(f"📄 Setup summary saved to: {project_dir}/setup_summary.md")

## 🎉 Setup Complete!

Congratulations! You've successfully set up the RLHF Phi-3 pipeline. Here's what you've accomplished:

✅ **Environment Setup**: Configured Google Colab with GPU access  
✅ **Storage**: Set up Google Drive for checkpoint persistence  
✅ **Authentication**: Configured W&B and HuggingFace access  
✅ **Installation**: Installed the pipeline and all dependencies  
✅ **Configuration**: Validated and customized pipeline settings  
✅ **Testing**: Verified all components are working correctly  

## 🚀 Next Steps

You're now ready to start training! Here's the recommended sequence:

1. **[SFT Training Tutorial](02_sft_training_tutorial.ipynb)** - Learn supervised fine-tuning
2. **[Reward Model Tutorial](03_reward_model_tutorial.ipynb)** - Train a reward model
3. **[PPO Training Tutorial](04_ppo_training_tutorial.ipynb)** - Complete RLHF training
4. **[Evaluation and Publishing](05_evaluation_and_publishing.ipynb)** - Evaluate and share your model

## 💡 Tips for Success

- **Monitor GPU usage**: Keep an eye on memory usage to avoid OOM errors
- **Save frequently**: Checkpoints are automatically saved, but manual saves are good practice
- **Watch session time**: Colab sessions timeout after 12 hours
- **Use streaming datasets**: For large datasets to avoid memory issues
- **Check logs**: W&B provides excellent training visualization

## 🆘 Need Help?

- **Documentation**: Check the README.md for detailed information
- **Troubleshooting**: Review the troubleshooting section for common issues
- **Community**: Join discussions on GitHub Issues

Happy training! 🚀